In [ ]:
import pyfar as pf
import numpy as np
import scipy as sc
import rdp
import matplotlib.pyplot as plt


guitar = pf.dsp.time_crop(pf.signals.files.guitar(), [1, 1.05], unit='s')
sweep = pf.signals.sine(13, 2205)

def reduce(x, y, max_mse):
    epsilon = 0.
    M = np.stack([x, y]).T
    epsilon = np.geomspace(0.00001, 0.001, 20)
    for e in epsilon:
        xp, yp = rdp.rdp(M, epsilon=e).T
        yr = np.interp(x, xp, yp)
        err = np.mean(np.pow(y - yr, 2))
        print(e, len(xp), err)
        if err >= max_mse:
            return xp, yp


def reduce_and_plot(x,y):
    ax = plt.subplot()
    # ax.plot(x, y, color='blue', alpha=0.5)
    reduced_x, reduced_y = reduce(x, y, 1e-7)
    print(x.shape, reduced_x.shape)
    ax.plot(reduced_x, reduced_y, color='red', alpha=0.5)
    plt.show()

reduce_and_plot(guitar.times, guitar.time[0])
reduce_and_plot(sweep.times, sweep.time[0])


In [ ]:
N = 1000
phase = np.linspace(0, np.pi * 2, N)
# blip = np.sin(phase * 8) * np.exp(1-phase)
blip = pf.signals.files.guitar().time[0,44100:44100+N]

def spline_reduce(x, y, max_mse):
    epsilon = 0.
    M = np.stack([x, y]).T
    epsilon = np.geomspace(0.00001, 0.01, 10)
    for e in epsilon:
        xp, yp = rdp.rdp(M, epsilon=e).T
        spline = sc.interpolate.make_splrep(xp, yp)
        yr = spline(x)
        err = np.mean(np.pow(y - yr, 2))
        print(e, len(xp), err)
        if err >= max_mse:
            return xp, yp, spline


def simplify(X, Y, max_mse):
    zero_crossings = np.argwhere(np.diff(np.sign(Y)) != 0).flatten()
    inflection_points = np.argwhere(np.diff(np.sign(np.diff(Y))) != 0).flatten()
    inflection_points_2 = np.argwhere(np.diff(np.sign(np.diff(Y, n=2))) != 0).flatten()

    important_points = np.hstack([
        [0],
        zero_crossings,
        inflection_points,
        inflection_points_2,
        [len(blip) - 1],
    ])
    important_points.sort()
    important_points = np.unique(important_points)

    return X[important_points], Y[important_points]

    # return spline_reduce(X[important_points], Y[important_points], max_mse)

reduced_x, reduced_y = simplify(phase, blip, 1e-6)
spline = sc.interpolate.make_splrep(reduced_x, reduced_y)

print(blip.shape, reduced_x.shape)

fig, ax = plt.subplots(2,1, figsize=(12,12))
ax[0].plot(phase, blip)
# ax.plot(reduced_x, reduced_y, 'g-')
interp_y = spline(phase)
linear_interp_y = np.interp(phase, reduced_x, reduced_y)
spline_err = np.abs(interp_y - blip)
linear_err = np.abs(linear_interp_y - blip)
ax[0].plot(phase, interp_y, 'g-')
ax[1].plot(phase, spline_err, 'k-')
ax[1].plot(phase, linear_err, 'r-')

In [ ]:
envelopes = np.load("code/kappa/test_envelopes.npz")
print(envelopes["time"].shape)
t = envelopes["time"]

def _plot_subbands(ax, times, signal):
    N = np.arange(len(signal))
    ax.plot(times, np.flip(signal, axis=0).T + N, color='k', linewidth=0.5)

fig, ax = plt.subplots(1,1, figsize=(10,10))

_plot_subbands(ax, envelopes["time"], envelopes["data"])

for n, env in enumerate(reversed(envelopes["data"])):
    # reduced_x, reduced_y, spline = spline_reduce(t, env, 1e-5)
    # reduced_x, reduced_y = reduce(t, env, 1e-5)
    M = np.stack([t, env * 0.1]).T  # give less weight to the Y coordinate because they're squashed so much
    reduced_x, reduced_y = rdp.rdp(M, epsilon=0.0001).T
    reduced_y *= 10 # restore
    print(env.shape, reduced_x.shape)
    # spline = sc.interpolate.make_splrep(reduced_x, reduced_y)
    # ax.plot(t, spline(t) + n, 'g-')
    ax.plot(reduced_x, reduced_y + n, color='green', linestyle='-')
    # ax.plot(reduced_x, reduced_y + n, 'ro', markersize=1)
    